# 62 · Evaluate RAG — browse the leaderboard, then judge one live

**A RAG system can only be trusted as far as it is measured.** Retrieval can pull the wrong
passages, generation can invent facts the passages never said, and neither failure shows up
in a demo that happened to ask an easy question. The lab runs an **evaluation harness** so
"is the answer any good?" stops being a vibe and becomes a number — one that is comparable
across models and across runs.

This notebook is one layer above that harness. It does two things:

1. **Browses the eval results** — the leaderboard the harness already produced: which
   model/backend scores best on which metric, averaged over a fixed exam. That lives in the
   mesh (Iceberg lakehouse + operational Postgres) and we read it through **Trino**, reusing
   the federation shape of notebook `20` but pointed entirely at the *eval* story.
2. **Computes one metric live** — takes a single real `(question, contexts, answer)` row and
   scores its **faithfulness** with the judge LLM, mirroring the harness's own prompt, so you
   see the eval *mechanism* end-to-end on one concrete example.

### The three metrics, in plain terms

Every answer is scored by an LLM **judge** on three axes, each a float **0.0–1.0**:

| metric | the question it asks | what a low score means |
|--------|----------------------|------------------------|
| **faithfulness** | is the ANSWER supported by the retrieved CONTEXT (no invented facts)? | the model hallucinated — said things the context didn't |
| **answer_relevancy** | does the ANSWER actually address the QUESTION? | the model drifted, dodged, or over-answered |
| **context_relevancy** | is the retrieved CONTEXT on-topic for the QUESTION? | *retrieval* failed — it fetched the wrong passages |

The first two grade **generation**; the third grades **retrieval**. Splitting them is the
point — a bad answer with good context is a generation problem, a bad answer with bad context
is a retrieval problem, and one blended score would hide which.

### The golden exam, and the model matrix

Scores only mean something if every run is graded on the **same exam**. The harness pins a
**golden question set** of **20 questions**, deliberately in two halves:

- **10 conceptual** — prose/reasoning questions (the ADKAR change model, affinity mapping,
  evaluating an AI provider for healthcare). Dense embeddings already do well here.
- **10 lexical** — exact identifiers pulled from the lab's own docs (a JVM heap size, a
  NodePort, an Ollama env var, a BM25 index name). This is where dense retrieval is weak and
  keyword (BM25) retrieval is strong — the half that *moves* when retrieval improves.

Each golden question is run through **every model** in the matrix
(`qwen3:30b-a3b`, `qwen3-coder:30b`, `deepseek-coder-v2:16b`, `gpt-oss:20b`, `qwen3:14b`,
`mistral-small3.2:24b`) against the `pgvector` retrieval backend, the answers are stored, and
then a **panel of judges** scores each answer. The leaderboard averages across the panel —
single-judge LLM-as-judge is noisy (one model went 5th→1st just by swapping the judge), so a
panel is used to damp that.

> **Read-only, throughout.** Everything here is `SELECT` against the eval stores and **one**
> live judge call that writes nothing — it scores an existing answer in memory and prints the
> number. Nothing is inserted into any store, and (like notebooks `20`/`22`) there is **no
> cleanup section** because we create nothing.

## Setup

The `trino` client (to read the leaderboard) and `openai` client (to call the judge gateway,
which is OpenAI-compatible) are **not** in the singleuser base image — which ships `polars`,
`s3fs`, `pyarrow`, `duckdb`, `fastavro` — so we install them here. `polars`, used to render
every result frame exactly as in notebooks `20`/`22`, already ships in the image.

In [1]:
%pip install -q trino openai


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connect Trino

Connection is **env-driven** with committed **in-cluster** defaults
(`trino.data-mesh.svc.cluster.local:8080`); a run outside the cluster overrides `TRINO_HOST` /
`TRINO_PORT` without editing the notebook — the pattern every query notebook uses. Trino here
speaks plain HTTP and takes an empty password. We pin no catalog, so a query can fully-qualify
names and cross catalogs (the eval leaderboard joins the lakehouse to the operational DB).

`q(sql)` runs a statement and hands the rows back as a **polars** DataFrame; `scalar(sql)`
returns the single top-left value.

In [2]:
import os
import trino
import polars as pl

TRINO_HOST = os.environ.get("TRINO_HOST", "trino.data-mesh.svc.cluster.local")
TRINO_PORT = int(os.environ.get("TRINO_PORT", "8080"))
TRINO_USER = os.environ.get("TRINO_USER", "jupyter")

conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user=TRINO_USER,
    http_scheme="http",   # this coordinator is plain HTTP, unmeshed
    # no catalog/schema pinned -> the eval leaderboard join can cross iceberg + postgresql
)

def q(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pl.DataFrame(rows, schema=cols, orient="row")

def scalar(sql):
    cur = conn.cursor()
    cur.execute(sql)
    return cur.fetchone()[0]

print("Trino host   :", f"{TRINO_HOST}:{TRINO_PORT}")
print("Trino version:", scalar("SELECT version()"))   # proves the connection
print("as user      :", TRINO_USER)

Trino host   : trino.data-mesh.svc.cluster.local:8080
Trino version: 468
as user      : jupyter


### Where the eval data lives

The harness writes to the **operational Postgres** as it runs, and a copy of the scores lands
in the **Iceberg lakehouse**. Four tables carry the story:

| table | catalog | what it holds |
|-------|---------|---------------|
| **`eval_runs`** | `postgresql.public` | one row per evaluation run — `status`, `question_count`, `notes` |
| **`eval_questions`** | `postgresql.public` | the exam for a run — the golden questions, one row each |
| **`eval_results`** | `postgresql.public` | one row per (run, question, model) — the `answer`, retrieved `contexts`, `backend`, `latency_ms` |
| **`eval_scores`** | `iceberg.eval` (+ `postgresql.public`) | one row per (result, metric, judge) — the judge's `score` |

The leaderboard is the **join** of `eval_scores` (the numbers) to `eval_results` (which model,
which backend, how slow). We read the lakehouse copy of the scores and join it to the live
operational results — exactly the cross-engine join notebook `20` demonstrates, here in service
of the eval question rather than to show federation for its own sake.

## The leaderboard

First the run ledger: `eval_runs.status` walks each run through `questions_ready` →
`results_ready` (answers generated) → `scored` (judged). We roll it up, then look at the most
recent scored run.

In [3]:
q('''
    SELECT status, count(*) AS n_runs, sum(question_count) AS total_questions
    FROM postgresql.public.eval_runs
    GROUP BY status
    ORDER BY n_runs DESC
''')

status,n_runs,total_questions
str,i64,i64
"""scored""",14,240
"""results_ready""",6,110
"""questions_ready""",3,50


In [4]:
q('''
    SELECT id, status, question_count, created_at
    FROM postgresql.public.eval_runs
    WHERE status = 'scored'
    ORDER BY id DESC
    LIMIT 5
''')

id,status,question_count,created_at
i64,str,i64,"datetime[μs, UTC]"
22,"""scored""",20,2026-08-22 07:00:11.616176 UTC
18,"""scored""",20,2026-08-08 07:00:20.344151 UTC
14,"""scored""",20,2026-07-28 04:03:35.279446 UTC
13,"""scored""",20,2026-07-27 19:33:49.089815 UTC
12,"""scored""",20,2026-07-27 14:57:47.342674 UTC


### Per model x backend x metric — the leaderboard join

The centerpiece. `iceberg.eval.eval_scores` (the lakehouse copy of the judge output)
contributes `metric` and `score`; `postgresql.public.eval_results` (the operational store)
contributes `model`, `backend`, and `latency_ms`. They meet on
`eval_scores.result_id = eval_results.id`. The average `score` is taken across the whole judge
**panel** and across all golden questions, so each cell is "how this model does on this metric,
over the fixed exam, as the panel sees it".

In [5]:
fed = q('''
    SELECT r.model,
           r.backend,
           s.metric,
           count(*)                    AS n_scores,
           round(avg(s.score), 3)      AS avg_score,
           round(avg(r.latency_ms), 0) AS avg_ms
    FROM iceberg.eval.eval_scores        s          -- lakehouse  (judge scores)
    JOIN postgresql.public.eval_results  r          -- operational (which model, how slow)
      ON s.result_id = r.id
    GROUP BY r.model, r.backend, s.metric
    ORDER BY r.model, s.metric
''')
print(f"{fed.height} (model x backend x metric) cells across iceberg + postgresql")
fed

18 (model x backend x metric) cells across iceberg + postgresql


model,backend,metric,n_scores,avg_score,avg_ms
str,str,str,i64,f64,f64
"""deepseek-coder-v2:16b""","""pgvector""","""answer_relevancy""",645,0.779,11027.0
"""deepseek-coder-v2:16b""","""pgvector""","""context_relevancy""",645,0.696,11027.0
"""deepseek-coder-v2:16b""","""pgvector""","""faithfulness""",645,0.71,11027.0
"""gpt-oss:20b""","""pgvector""","""answer_relevancy""",657,0.785,15428.0
"""gpt-oss:20b""","""pgvector""","""context_relevancy""",655,0.671,15438.0
…,…,…,…,…,…
"""qwen3:14b""","""pgvector""","""context_relevancy""",647,0.679,31531.0
"""qwen3:14b""","""pgvector""","""faithfulness""",647,0.747,31531.0
"""qwen3:30b-a3b""","""pgvector""","""answer_relevancy""",669,0.763,31397.0


Read wide, one row per model, one column per metric — the leaderboard as you'd actually
scan it. `context_relevancy` (retrieval quality) tends to sit a little below the two
generation metrics across the board, which is the honest shape: retrieval is the harder half.

In [6]:
# wide view: model x backend down the side, the three metrics across the top.
# (the pivot keyword changed across polars versions -> try both so the run never errors)
try:
    lb_wide = fed.pivot(values="avg_score", index=["model", "backend"], on="metric")
except TypeError:
    lb_wide = fed.pivot(values="avg_score", index=["model", "backend"], columns="metric")
lb_wide

model,backend,answer_relevancy,context_relevancy,faithfulness
str,str,f64,f64,f64
"""deepseek-coder-v2:16b""","""pgvector""",0.779,0.696,0.71
"""gpt-oss:20b""","""pgvector""",0.785,0.671,0.744
"""mistral-small3.2:24b""","""pgvector""",0.749,0.642,0.701
"""qwen3-coder:30b""","""pgvector""",0.779,0.686,0.746
"""qwen3:14b""","""pgvector""",0.796,0.679,0.747
"""qwen3:30b-a3b""","""pgvector""",0.763,0.654,0.72


And the headline: **the best model on each metric**. We sort by score and take the top
model per metric — the single fact a leaderboard exists to answer.

In [7]:
best = (
    fed.sort("avg_score", descending=True)
       .group_by("metric", maintain_order=True)
       .first()
       .select(["metric", "model", "backend", "avg_score"])
       .sort("metric")
)
best

metric,model,backend,avg_score
str,str,str,f64
"""answer_relevancy""","""qwen3:14b""","""pgvector""",0.796
"""context_relevancy""","""deepseek-coder-v2:16b""","""pgvector""",0.696
"""faithfulness""","""qwen3:14b""","""pgvector""",0.747


## Drill into one result

An average hides the individual answers it is built from. Before trusting the leaderboard, look
at **one real row** the way the judge saw it: the golden question, the passages retrieval
fetched (`contexts`), and the model's `answer`. From the **most recent scored run**, we pick —
deterministically — the answer the judge panel rated **highest on faithfulness** (with a real,
substantive answer, not a one-line refusal), joining `eval_results` to `eval_questions` for the
question text. A well-supported answer makes the clearest example: we can watch the live judge
below arrive at the same verdict the panel did.

In [8]:
row = q('''
    WITH latest_scored AS (
        SELECT max(id) AS run_id FROM postgresql.public.eval_runs WHERE status = 'scored'
    ),
    faith AS (   -- panel-average faithfulness per result (the metric we judge live below)
        SELECT result_id, avg(score) AS avg_faith
        FROM iceberg.eval.eval_scores
        WHERE metric = 'faithfulness'
        GROUP BY result_id
    )
    SELECT r.id                     AS result_id,
           r.run_id,
           r.model,
           r.backend,
           eq.question,
           r.answer,
           json_format(r.contexts) AS contexts_json,
           r.latency_ms,
           round(f.avg_faith, 3)    AS panel_faith
    FROM postgresql.public.eval_results r
    JOIN postgresql.public.eval_questions eq ON eq.id = r.question_id
    JOIN faith         f ON f.result_id = r.id
    JOIN latest_scored l ON r.run_id    = l.run_id     -- most recent scored (golden) run
    WHERE r.error IS NULL
      AND r.answer   IS NOT NULL
      AND r.contexts IS NOT NULL
      AND json_format(r.contexts) NOT IN ('null', '[]')
      AND length(r.answer) > 200                       -- a real answer, not a one-line refusal
    ORDER BY f.avg_faith DESC, r.id                    -- a well-answered question, deterministically
    LIMIT 1
''').row(0, named=True)

RESULT_ID = row["result_id"]
QUESTION  = row["question"]
ANSWER    = row["answer"]

print("result_id :", RESULT_ID, "| run", row["run_id"], "| model", row["model"],
      "| backend", row["backend"], "|", row["latency_ms"], "ms")
print("panel faithfulness (stored):", row["panel_faith"])
print()
print("QUESTION  :", QUESTION)
print()
print("ANSWER    :", (ANSWER[:700] + " ...") if len(ANSWER) > 700 else ANSWER)

result_id : 2112 | run 22 | model qwen3:30b-a3b | backend pgvector | 96293 ms
panel faithfulness (stored): 1.0

QUESTION  : Which Traefik setting had to be changed to stop large image pushes to the registry failing at 60 seconds?

ANSWER    : Based on the provided context, the Traefik setting that needed to be changed was `readTimeout`.

From context chunk [1] (traefik-config.yaml), specifically the comment section:
"B69 Wave 3 — ROOT CAUSE (proven 2026-07-17): Traefik v3's default `readTimeout` (60s) bounds reading the ENTIRE request body. Pushing weyland-tool-server (a single ~3GB layer) to registry.weyland.lab streams for >60s, so Traefik guillotined the request at exactly 60s → HTTP 499, and the registry pod never received the blob."

The fix was detailed in the configuration:
"readTimeout=0 removes the read deadline on the shared HTTPS entrypoint (and HTTP, for completeness)."

The configuration in the YAML file shows:
```
p ...


The `contexts` are the passages retrieval handed the model — stored as JSON. Each source
carries the retrieved `content` (and provenance). These are exactly the passages the
faithfulness judge checks the answer against: an answer is faithful only if everything it
claims is supported *here*.

In [9]:
import json

# eval_results.contexts is JSON text (a list of retrieved sources); parse it as the harness does.
try:
    CONTEXTS = json.loads(row["contexts_json"]) or []
except (TypeError, json.JSONDecodeError):
    CONTEXTS = []

print(f"{len(CONTEXTS)} retrieved context passage(s):\n")
for i, c in enumerate(CONTEXTS, 1):
    content = c.get("content", "") if isinstance(c, dict) else str(c)
    snippet = content.replace("\n", " ").strip()
    print(f"[{i}] {(snippet[:280] + ' ...') if len(snippet) > 280 else snippet}\n")

3 retrieved context passage(s):

[1] # k3s bundles Traefik and reconciles it via its built-in helm-controller. To override the bundled chart's values # we apply a HelmChartConfig named `traefik` in kube-system — the helm-controller MERGES valuesContent over the # chart defaults and redeploys Traefik. # # B69 Wave 3  ...

[2] ther**).  Gotchas: - After editing manifests, always **remind the user to push** and name the exact file paths — an un-pushed manifest never deploys. - CRDs / configs over ~256KB exceed the last-applied-annotation limit → add `argocd.argoproj.io/sync-options: ServerSideApply=true ...

[3] image: registry:2.8.3  # nosemgrep: yaml.kubernetes.security.run-as-non-root.run-as-non-root           imagePullPolicy: IfNotPresent           # B47 hardening: drop priv-esc + seccomp (see arch)           securityContext:             allowPrivilegeEscalation: false             se ...



## Judge it live — mirror the harness's faithfulness prompt

Now the mechanism itself, on this one row. The offline harness scores each answer by sending
the judge a **strict-evaluator prompt** — question + context + answer → a JSON object of three
0.0–1.0 floats — with `response_format = json_object` so the reply is parseable. We reproduce
**that exact prompt** (from the harness's own `_judge`), send it to the LiteLLM gateway judge
alias **`wl-judge`** (OpenAI-compatible), parse the score, and compare our live
**faithfulness** number to the one the offline panel already stored for this result.

Two honest caveats, both expected:

- The offline leaderboard averages a **panel** of judges; this is **one** judge via the
  gateway. Expect a *ballpark* match, not an identical number — that variance is exactly why
  the harness uses a panel.
- The whole call is wrapped in `try/except`. A judge that is slow, offline, or returns
  unparseable text **degrades to a printed note**, never a notebook error — the eval harness
  itself fails safe the same way (a judge hiccup must never sink a run).

Connection is env-driven with the in-cluster gateway default; the key comes from the
`LITELLM_MASTER_KEY` environment variable and is never written into the notebook.

In [10]:
import re

LITELLM_BASE  = os.environ.get("LITELLM_BASE_URL", "http://litellm.weyland.svc.cluster.local:4000/v1")
JUDGE_MODEL   = os.environ.get("JUDGE_MODEL", "wl-judge")   # LiteLLM alias; wl-judge-oss is the OSS fallback
METRICS       = ("faithfulness", "answer_relevancy", "context_relevancy")

def _strip_think(text):
    # judges may emit a <think>...</think> block; the harness strips it before parsing (eval_scores.py)
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def build_judge_prompt(question, contexts, answer):
    # verbatim shape of the harness _judge prompt (weyland_pipeline/assets/eval_scores.py)
    ctx = "\n\n".join(
        (c.get("content", "") if isinstance(c, dict) else str(c)) for c in (contexts or [])
    )[:8000]
    return (
        "You are a strict RAG evaluator. Score the ANSWER on three metrics, each a float 0.0-1.0:\n"
        "- faithfulness: is the ANSWER supported by the CONTEXT (no invented facts)?\n"
        "- answer_relevancy: does the ANSWER address the QUESTION?\n"
        "- context_relevancy: is the CONTEXT relevant to the QUESTION?\n"
        'Respond ONLY with JSON: {"faithfulness":0.0,"answer_relevancy":0.0,"context_relevancy":0.0}\n\n'
        f"QUESTION:\n{question}\n\nCONTEXT:\n{ctx}\n\nANSWER:\n{_strip_think(answer)}"
    )

def parse_scores(content):
    # robust parse: strip think, then json.loads, then fall back to the first {...} block
    content = _strip_think(content)
    try:
        obj = json.loads(content)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", content, flags=re.DOTALL)
        obj = json.loads(m.group(0)) if m else {}
    return {k: float(obj[k]) for k in METRICS if k in obj}

prompt = build_judge_prompt(QUESTION, CONTEXTS, ANSWER)
print(f"judge alias : {JUDGE_MODEL}  (gateway: {LITELLM_BASE})")
print(f"prompt built: {len(prompt)} chars  ({len(CONTEXTS)} context passage(s) folded in)")

judge alias : wl-judge  (gateway: http://litellm.weyland.svc.cluster.local:4000/v1)
prompt built: 6014 chars  (3 context passage(s) folded in)


In [11]:
from openai import OpenAI

live_scores, live_note = {}, None
try:
    judge = OpenAI(base_url=LITELLM_BASE, api_key=os.environ["LITELLM_MASTER_KEY"])
    resp = judge.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},   # same constraint the harness uses
        stream=False,
    )
    live_scores = parse_scores(resp.choices[0].message.content or "")
    if not live_scores:
        live_note = "judge replied but no parseable metric floats were found"
except KeyError:
    live_note = "LITELLM_MASTER_KEY is not set in this environment — skipping the live judge call"
except Exception as exc:  # slow/offline gateway, bad JSON, etc. -> degrade to a note, never an error output
    live_note = f"live judge call did not complete: {type(exc).__name__}: {str(exc)[:200]}"

if live_scores:
    print("live judge scores (this one answer, one judge):")
    for m in METRICS:
        if m in live_scores:
            print(f"  {m:18s}: {live_scores[m]:.3f}")
else:
    print("NOTE:", live_note)

live judge scores (this one answer, one judge):
  faithfulness      : 0.800
  answer_relevancy  : 1.000
  context_relevancy : 1.000


### Live vs stored — does the mechanism agree?

The offline panel already scored this result. We pull the **stored faithfulness** for this
`result_id` (the per-judge rows and their average) and set our live number beside it. A close
match says the mechanism is reproducible; a gap is the panel-vs-single-judge variance the
harness is built to average away — either way, this is the eval loop shown end-to-end on one
concrete example.

In [12]:
stored = q(f'''
    SELECT judge, round(score, 3) AS stored_score
    FROM iceberg.eval.eval_scores
    WHERE result_id = {RESULT_ID} AND metric = 'faithfulness'
    ORDER BY judge
''')
print(f"stored faithfulness for result_id {RESULT_ID} (per judge in the panel):")
stored

stored faithfulness for result_id 2112 (per judge in the panel):


judge,stored_score
str,f64
"""deepseek-coder-v2:16b""",1.0
"""mistral-small3.2:24b""",1.0
"""qwen3-coder:30b""",1.0


In [13]:
stored_avg = q(f'''
    SELECT round(avg(score), 3) AS panel_avg_faithfulness, count(*) AS n_judges
    FROM iceberg.eval.eval_scores
    WHERE result_id = {RESULT_ID} AND metric = 'faithfulness'
''').row(0, named=True)

panel_avg = stored_avg["panel_avg_faithfulness"]
live_f    = live_scores.get("faithfulness")

print(f"stored (panel avg, {stored_avg['n_judges']} judges): {panel_avg}")
if live_f is not None:
    print(f"live   (wl-judge, this run)        : {live_f:.3f}")
    print(f"delta                              : {abs(live_f - float(panel_avg)):.3f}")
else:
    print("live   (wl-judge, this run)        : n/a —", live_note)
    print("(the stored panel score stands on its own; the live call is illustrative)")

stored (panel avg, 3 judges): 1.0
live   (wl-judge, this run)        : 0.800
delta                              : 0.200


## When to reach for eval

**Two eval surfaces exist in the lab, for two different jobs — this notebook is a window into
the first and a hand-cranked taste of the second.**

- **Offline batch eval (the Dagster harness)** — the leaderboard above. It runs the *whole*
  golden exam across *every* model, judged by a *panel*, on a schedule, and pins the questions
  so a score change means the **system** changed, not the exam. Reach for it to **compare
  models**, to **prove a retrieval change helped** (watch the lexical half move), and to catch
  regressions over time. Its scores also surface in the eval experiment on **MLflow** and, for
  the online lane, in **Langfuse** — this notebook reads the results those runs deposited.

- **Ad-hoc single-example judge (this notebook, section above)** — one `(question, contexts,
  answer)` scored by one judge, right now. Reach for it to **debug a specific bad answer**
  ("is this a retrieval miss or a hallucination?" → look at `context_relevancy` vs
  `faithfulness`), to **sanity-check the judge prompt** before trusting a batch, or to
  understand the mechanism. It is not a benchmark — one example, one judge, no pinned exam.

| you want to… | reach for |
|--------------|-----------|
| rank models / prove a change helped / catch regressions | **offline batch eval** (Dagster harness → this leaderboard) |
| understand *why* one answer scored the way it did | **drill in + judge it live** (this notebook, sections 4–5) |
| the raw scores to build your own view over | **Trino** over `iceberg.eval.eval_scores` + `postgresql.public.eval_results` |

The rule of thumb: **the batch harness tells you *which* system is better; a live single-example
judge tells you *why* a given answer is good or bad.** You browse the first to decide, and reach
for the second when a number on the leaderboard surprises you and you need to see the answer
behind it.